# Decoding Separation 실험: "잔차(Residual)가 배경을 흡수했는가?"

## V-CoRes 모델의 Concept-Residual 분리 분석 (CUB-200-2011)

### 실험 목적
V-CoRes 모델의 핵심 가설을 검증합니다:
- **VariationalConceptBranch**는 새의 부리 형태, 날개 색상, 가슴 무늬 등 **의미적 개념**을 학습
- **ResidualBranch**는 배경, 조명, 포즈, 노이즈 등 **개념 외 정보**를 학습

### 실험 방법
1. 이미지를 인코딩하여 $z_{concept}$과 $z_{residual}$을 추출
2. **개념만 디코딩**: $z_{total} = \text{AggLayer}(z_{concept}, \mathbf{0})$ → 배경 없는 프로토타입 이미지
3. **잔차만 디코딩**: $z_{total} = \text{AggLayer}(\mathbf{0}, z_{residual})$ → 개념 없는 배경/디테일 이미지
4. **전체 디코딩**: $z_{total} = \text{AggLayer}(z_{concept}, z_{residual})$ → 완전한 재구성

### 기대 결과
- Concept-only 이미지: 새의 형태와 패턴이 드러나고, 배경은 단조롭거나 뭉개짐
- Residual-only 이미지: 새의 형태가 사라지고, 배경/조명/포즈 정보만 복원
- 이를 통해 V-CoRes가 Information Leakage 없이 순수한 개념 분리를 달성함을 증명

## 1. 환경 설정

In [2]:
import os
import sys
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

# 프로젝트 루트를 path에 추가
PROJECT_ROOT = Path(os.getcwd()).parent if "notebooks" in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from models.vcores import VCoResModel, VCoResLoss
from data.cub200 import CUB200Dataset, get_cub200_loaders

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

Using device: cuda
Project root: /home/dojan/workspace/cores-embedding


## 2. CUB-200-2011 데이터셋 준비

CUB-200-2011은 200종의 북미 조류 이미지 11,788장과 312개의 이진 속성 라벨을 제공합니다.

**자동 다운로드**가 지원됩니다. 이미 다운로드된 경우 건너뜁니다.

In [3]:
# === CUB-200-2011 다운로드 및 압축 해제 ===
CUB_URL = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
DATA_ROOT = PROJECT_ROOT / "data" / "raw"
CUB_DIR = DATA_ROOT / "cub200" / "CUB_200_2011"

def download_cub200(data_root: Path, url: str = CUB_URL):
    """CUB-200-2011 데이터셋을 자동 다운로드하고 압축 해제합니다."""
    cub_parent = data_root / "cub200"
    cub_dir = cub_parent / "CUB_200_2011"

    if cub_dir.exists() and (cub_dir / "images").exists():
        print(f"[CUB-200] 이미 존재: {cub_dir}")
        return cub_dir

    cub_parent.mkdir(parents=True, exist_ok=True)
    tgz_path = cub_parent / "CUB_200_2011.tgz"

    if not tgz_path.exists():
        print(f"[CUB-200] 다운로드 중... ({url})")
        print("  (~1.1 GB, 수 분 소요될 수 있습니다)")

        def _progress(block_num, block_size, total_size):
            downloaded = block_num * block_size
            pct = downloaded / total_size * 100 if total_size > 0 else 0
            mb = downloaded / 1e6
            print(f"\r  {mb:.1f} MB ({pct:.1f}%)", end="", flush=True)

        urllib.request.urlretrieve(url, str(tgz_path), reporthook=_progress)
        print("\n  다운로드 완료!")
    else:
        print(f"[CUB-200] 아카이브 존재: {tgz_path}")

    print("[CUB-200] 압축 해제 중...")
    with tarfile.open(str(tgz_path), "r:gz") as tar:
        tar.extractall(path=str(cub_parent))
    print("  압축 해제 완료!")

    return cub_dir

cub_dir = download_cub200(DATA_ROOT)
print(f"\nCUB-200 경로: {cub_dir}")
print(f"이미지 디렉토리 존재: {(cub_dir / 'images').exists()}")

[CUB-200] 다운로드 중... (https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz)
  (~1.1 GB, 수 분 소요될 수 있습니다)
  1150.6 MB (100.0%)
  다운로드 완료!
[CUB-200] 압축 해제 중...
  압축 해제 완료!

CUB-200 경로: /home/dojan/workspace/cores-embedding/data/raw/cub200/CUB_200_2011
이미지 디렉토리 존재: True


In [ ]:
# === 데이터셋 및 DataLoader 구성 ===
IMG_SIZE = 64
NUM_CONCEPTS = 20  # 사용할 개념 속성 수 (312개 중 가장 균형 잡힌 상위 20개)
BATCH_SIZE = 64

# 시각화용 transform (정규화 포함)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 8, IMG_SIZE + 8)),
    transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# 시각화 전용 transform (정규화 없이 원본 색상 유지)
vis_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # [0, 1] 범위
])

# 학습용 데이터셋
train_dataset = CUB200Dataset(
    root=str(DATA_ROOT), split="train",
    transform=train_transform, num_concepts=NUM_CONCEPTS,
)
test_dataset = CUB200Dataset(
    root=str(DATA_ROOT), split="test",
    transform=test_transform, num_concepts=NUM_CONCEPTS,
)

# 시각화용 데이터셋 (정규화 없음 — 디코더 출력 [0,1]과 비교용)
vis_dataset = CUB200Dataset(
    root=str(DATA_ROOT), split="test",
    transform=vis_transform, num_concepts=NUM_CONCEPTS,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
)

print(f"\n학습 데이터: {len(train_dataset)}장")
print(f"테스트 데이터: {len(test_dataset)}장")
print(f"개념 수: {train_dataset.num_concepts}")
print(f"개념 속성: {train_dataset.get_attribute_names()[:10]}...")

[CUB-200] Loaded 5994 train images, 20 concepts, 200 classes
[CUB-200] Loaded 5794 test images, 20 concepts, 200 classes


In [ ]:
# === 샘플 이미지 미리보기 ===
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("CUB-200-2011 샘플 이미지", fontsize=14, fontweight="bold")

for i, ax in enumerate(axes.flat):
    img, concepts, class_label = vis_dataset[i * 50]
    ax.imshow(img.permute(1, 2, 0).numpy())
    active = [train_dataset.get_attribute_names()[j]
              for j in range(len(concepts)) if concepts[j] > 0.5]
    title = f"Class {class_label.item()}"
    if active:
        title += f"\n{', '.join(active[:2])}"
    ax.set_title(title, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. V-CoRes 모델 구성 및 학습

CUB-200 데이터셋에 맞추어 V-CoRes 모델을 초기화하고 학습합니다.

### 모델 아키텍처
```
ResNet-18 (shared) → ┌→ VariationalConceptBranch → z_concept (sampled)
                      └→ ResidualBranch           → z_residual
                      → AggregationLayer           → z_total ∈ R^D
                      → LatentDecoder              → x̂ (reconstruction)
```

In [ ]:
# === V-CoRes 모델 초기화 ===
LATENT_DIM = 64
CONCEPT_DIM = 32
RESIDUAL_DIM = 32

model = VCoResModel(
    latent_dim=LATENT_DIM,
    num_concepts=NUM_CONCEPTS,
    concept_dim=CONCEPT_DIM,
    residual_dim=RESIDUAL_DIM,
    use_soft_concepts=True,
    concept_temperature=1.0,
    aggregation="sum",
    image_size=IMG_SIZE,
    use_decoder=True,
).to(device)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"V-CoRes 모델 초기화 완료")
print(f"  총 파라미터:   {total_params:,}")
print(f"  학습 파라미터: {trainable_params:,}")
print(f"  Concept dim:   {CONCEPT_DIM}")
print(f"  Residual dim:  {RESIDUAL_DIM}")
print(f"  Latent dim D:  {LATENT_DIM}")
print(f"  Decoder:       {'활성화' if model.use_decoder else '비활성화'}")

In [ ]:
# === 기존 체크포인트 로드 (있는 경우) 또는 학습 시작 ===
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "vcores_cub200" / "checkpoints"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best.pt"

TRAIN_FROM_SCRATCH = not CHECKPOINT_PATH.exists()

if not TRAIN_FROM_SCRATCH:
    print(f"[체크포인트 로드] {CHECKPOINT_PATH}")
    ckpt = torch.load(str(CHECKPOINT_PATH), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
    print("  모델 로드 완료!")
else:
    print("체크포인트가 없습니다. 학습을 시작합니다.")
    print(f"  체크포인트 저장 위치: {CHECKPOINT_DIR}")

In [ ]:
# === 학습 루프 ===
# 이미 체크포인트가 있으면 건너뜁니다.
# 재구성(Reconstruction) 품질이 핵심이므로 충분한 에포크 학습을 권장합니다.

NUM_EPOCHS = 30  # 필요에 따라 조정 (권장: 50~100)
LR = 3e-4
KL_ANNEALING_EPOCHS = 10

if TRAIN_FROM_SCRATCH:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    criterion = VCoResLoss(
        concept_weight=1.0,
        recon_weight=2.0,      # 재구성 가중치를 높여 디코더 품질 확보
        kl_weight=0.5,         # β-VAE: KL 페널티 조절
        residual_reg_weight=0.01,
        orthogonality_weight=0.1,
        kl_annealing_epochs=KL_ANNEALING_EPOCHS,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6,
    )

    # ImageNet 역정규화 → [0, 1] 범위 (디코더 출력이 Sigmoid이므로)
    # 학습 시 원본 이미지를 [0,1]로 변환하여 recon loss 계산
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225],
    )

    best_loss = float("inf")
    history = {"loss_total": [], "loss_recon": [], "loss_kl": [],
               "loss_concept": [], "loss_orthogonality": []}

    model.train()
    for epoch in range(NUM_EPOCHS):
        epoch_losses = {}
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)

        for images, concept_labels, _ in pbar:
            images = images.to(device)
            concept_labels = concept_labels.to(device)

            # Forward
            output = model(images)

            # 재구성 타겟: 정규화 이미지를 역정규화하여 [0,1] 범위로 변환
            images_01 = torch.stack([inv_normalize(img) for img in images])
            images_01 = images_01.clamp(0, 1)

            loss, loss_dict = criterion(
                output, concept_labels, x_input=images_01, epoch=epoch,
            )

            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            for k, v in loss_dict.items():
                epoch_losses[k] = epoch_losses.get(k, 0) + v

            pbar.set_postfix({
                "loss": f"{loss_dict['loss_total']:.4f}",
                "recon": f"{loss_dict['loss_recon']:.4f}",
                "kl": f"{loss_dict['loss_kl']:.4f}",
            })

        scheduler.step()

        # 에포크 평균 로스
        n_batches = len(train_loader)
        avg_total = epoch_losses.get("loss_total", 0) / n_batches
        avg_recon = epoch_losses.get("loss_recon", 0) / n_batches
        avg_kl = epoch_losses.get("loss_kl", 0) / n_batches
        avg_concept = epoch_losses.get("loss_concept", 0) / n_batches
        avg_orth = epoch_losses.get("loss_orthogonality", 0) / n_batches

        history["loss_total"].append(avg_total)
        history["loss_recon"].append(avg_recon)
        history["loss_kl"].append(avg_kl)
        history["loss_concept"].append(avg_concept)
        history["loss_orthogonality"].append(avg_orth)

        print(f"Epoch {epoch+1:3d} | Total: {avg_total:.4f} | "
              f"Recon: {avg_recon:.4f} | KL: {avg_kl:.4f} | "
              f"Concept: {avg_concept:.4f} | Orth: {avg_orth:.4f}")

        # 체크포인트 저장
        if avg_total < best_loss:
            best_loss = avg_total
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": best_loss,
            }, str(CHECKPOINT_PATH))
            print(f"  → Best model 저장 (loss={best_loss:.4f})")

        if (epoch + 1) % 10 == 0:
            ep_path = CHECKPOINT_DIR / f"epoch_{epoch+1}.pt"
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "loss": avg_total,
            }, str(ep_path))

    print("\n학습 완료!")
else:
    print("이미 학습된 모델을 로드했습니다. 학습을 건너뜁니다.")
    history = None

In [ ]:
# === 학습 커브 시각화 ===
if history is not None and len(history["loss_total"]) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history["loss_total"], "b-", linewidth=2, label="Total")
    axes[0].set_title("Total Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["loss_recon"], "r-", linewidth=2, label="Recon")
    axes[1].plot(history["loss_kl"], "g-", linewidth=2, label="KL")
    axes[1].set_title("Reconstruction & KL Loss", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(history["loss_concept"], "m-", linewidth=2, label="Concept")
    axes[2].plot(history["loss_orthogonality"], "c-", linewidth=2, label="Orthogonality")
    axes[2].set_title("Concept & Orthogonality Loss", fontweight="bold")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.suptitle("V-CoRes 학습 커브 (CUB-200)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("학습 히스토리가 없습니다 (기존 체크포인트에서 로드됨).")

## 4. Decoding Separation 실험

### 핵심 아이디어

V-CoRes의 AggregationLayer ("sum" 모드)는 다음과 같이 동작합니다:

$$z_{total} = W_c \cdot z_{concept} + b_c + W_r \cdot z_{residual} + b_r$$

이를 분리하면:

| 모드 | $z_{concept}$ | $z_{residual}$ | 결과 |
|------|:---:|:---:|------|
| **Full** | ✓ | ✓ | 완전한 재구성 |
| **Concept-only** | ✓ | $\mathbf{0}$ | 개념적 의미만 (새의 형태/색상) |
| **Residual-only** | $\mathbf{0}$ | ✓ | 배경/조명/포즈만 |

In [ ]:
# === Decoding Separation 핵심 함수 ===

@torch.no_grad()
def decode_separated(model, images, device):
    """이미지를 인코딩 후 concept-only, residual-only, full 디코딩을 수행합니다.

    Args:
        model: 학습된 VCoResModel (eval 모드).
        images: 입력 이미지 텐서 [B, 3, H, W] (정규화된 상태).
        device: torch device.

    Returns:
        dict with keys:
            'full_recon':     전체 재구성     [B, 3, H, W]
            'concept_only':   개념만 디코딩   [B, 3, H, W]
            'residual_only':  잔차만 디코딩   [B, 3, H, W]
            'concept_probs':  개념 활성화 확률 [B, N]
            'z_concept':      개념 임베딩      [B, concept_dim]
            'z_residual':     잔차 임베딩      [B, residual_dim]
    """
    model.eval()
    images = images.to(device)

    # 1. 인코딩: z_concept과 z_residual 추출
    (z_total, z_concept, z_residual,
     concept_probs, concept_logits, kl_loss, var_info) = model.encode(images)

    # 2. Full reconstruction (z_total 그대로 사용)
    full_recon = model.decode(z_total)

    # 3. Concept-only: z_residual = 0
    z_residual_zero = torch.zeros_like(z_residual)
    z_concept_only = model.aggregation(z_concept, z_residual_zero)
    concept_only_recon = model.decode(z_concept_only)

    # 4. Residual-only: z_concept = 0
    z_concept_zero = torch.zeros_like(z_concept)
    z_residual_only = model.aggregation(z_concept_zero, z_residual)
    residual_only_recon = model.decode(z_residual_only)

    return {
        "full_recon": full_recon.cpu(),
        "concept_only": concept_only_recon.cpu(),
        "residual_only": residual_only_recon.cpu(),
        "concept_probs": concept_probs.cpu(),
        "z_concept": z_concept.cpu(),
        "z_residual": z_residual.cpu(),
        "z_total": z_total.cpu(),
    }


def denormalize_imagenet(tensor):
    """ImageNet 정규화를 역변환하여 [0, 1] 범위로 복원합니다."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


def to_numpy_img(tensor):
    """Tensor [C, H, W] → NumPy [H, W, C] for matplotlib."""
    return tensor.permute(1, 2, 0).numpy().clip(0, 1)


print("Decoding separation 함수 정의 완료.")

## 5. 시각화: Concept vs Residual 분리 결과

In [ ]:
# === 테스트 이미지 선택 및 디코딩 분리 실행 ===
NUM_SAMPLES = 8  # 시각화할 이미지 수

# 다양한 종(class)에서 샘플링
np.random.seed(42)
sample_indices = np.random.choice(len(test_dataset), NUM_SAMPLES, replace=False)

# 정규화된 이미지 (모델 입력용)
sample_images = torch.stack([test_dataset[i][0] for i in sample_indices])
sample_concepts = torch.stack([test_dataset[i][1] for i in sample_indices])

# 원본 이미지 (시각화 비교용, 정규화 없음)
sample_originals = torch.stack([vis_dataset[i][0] for i in sample_indices])

# 디코딩 분리 실행
results = decode_separated(model, sample_images, device)

print(f"디코딩 분리 완료: {NUM_SAMPLES}장의 이미지")
print(f"  z_concept shape:  {results['z_concept'].shape}")
print(f"  z_residual shape: {results['z_residual'].shape}")
print(f"  z_total shape:    {results['z_total'].shape}")

In [ ]:
# === 메인 시각화: 4-row Comparison Grid ===

fig, axes = plt.subplots(4, NUM_SAMPLES, figsize=(NUM_SAMPLES * 2.5, 10))

row_labels = [
    "원본 (Original)",
    "전체 재구성 (Full)",
    "개념만 (Concept-only)",
    "잔차만 (Residual-only)",
]

for i in range(NUM_SAMPLES):
    # Row 0: 원본 이미지
    axes[0, i].imshow(to_numpy_img(sample_originals[i]))

    # Row 1: Full reconstruction
    axes[1, i].imshow(to_numpy_img(results["full_recon"][i]))

    # Row 2: Concept-only
    axes[2, i].imshow(to_numpy_img(results["concept_only"][i]))

    # Row 3: Residual-only
    axes[3, i].imshow(to_numpy_img(results["residual_only"][i]))

    # 상단에 활성 개념 표시
    probs = results["concept_probs"][i]
    top_k = torch.topk(probs, k=min(3, len(probs)))
    attr_names = train_dataset.get_attribute_names()
    top_concepts = [f"{attr_names[j]}({probs[j]:.1f})"
                    for j in top_k.indices if probs[j] > 0.3]
    axes[0, i].set_title(
        "\n".join(top_concepts[:2]) if top_concepts else "(no active)",
        fontsize=7,
    )

# 열 레이블
for row_idx, label in enumerate(row_labels):
    axes[row_idx, 0].set_ylabel(label, fontsize=11, fontweight="bold",
                                 rotation=0, labelpad=120, va="center")

for ax in axes.flat:
    ax.axis("off")

fig.suptitle(
    "V-CoRes Decoding Separation: Concept vs Residual\n"
    "개념 브랜치는 새의 의미적 패턴을, 잔차 브랜치는 배경/디테일을 캡처",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()

# 저장
save_dir = PROJECT_ROOT / "outputs" / "vcores_cub200" / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / "decoding_separation_grid.png"),
            dpi=150, bbox_inches="tight")
print(f"저장: {save_dir / 'decoding_separation_grid.png'}")
plt.show()

In [ ]:
# === 상세 분석: 개별 이미지 확대 비교 ===

def plot_single_separation(idx, model, test_dataset, vis_dataset, device,
                           save_dir=None):
    """단일 이미지에 대한 상세 Decoding Separation 시각화.

    원본, 전체 재구성, 개념만, 잔차만을 확대하여 보여주고,
    차이(difference) 맵과 개념 활성화 바 차트도 함께 표시합니다.
    """
    img_norm, concepts, class_label = test_dataset[idx]
    img_vis, _, _ = vis_dataset[idx]

    results = decode_separated(
        model, img_norm.unsqueeze(0), device
    )

    fig = plt.figure(figsize=(20, 8))
    gs = gridspec.GridSpec(2, 5, height_ratios=[3, 1], hspace=0.3, wspace=0.3)

    # --- Top row: 이미지 비교 ---
    titles_top = ["원본", "전체 재구성", "개념만", "잔차만", "차이맵\n(Concept - Residual)"]
    images_top = [
        to_numpy_img(img_vis),
        to_numpy_img(results["full_recon"][0]),
        to_numpy_img(results["concept_only"][0]),
        to_numpy_img(results["residual_only"][0]),
        None,  # 차이맵은 별도 처리
    ]

    for col in range(5):
        ax = fig.add_subplot(gs[0, col])
        if col < 4:
            ax.imshow(images_top[col])
        else:
            # 차이맵: concept_only - residual_only (절대값)
            diff = np.abs(
                to_numpy_img(results["concept_only"][0])
                - to_numpy_img(results["residual_only"][0])
            ).mean(axis=-1)
            im = ax.imshow(diff, cmap="hot", vmin=0, vmax=0.5)
            plt.colorbar(im, ax=ax, fraction=0.046)
        ax.set_title(titles_top[col], fontsize=12, fontweight="bold")
        ax.axis("off")

    # --- Bottom row: 개념 활성화 & 임베딩 분석 ---
    # (a) 개념 활성화 확률 바 차트
    ax_bar = fig.add_subplot(gs[1, :3])
    probs = results["concept_probs"][0].numpy()
    attr_names = train_dataset.get_attribute_names()
    colors = ["#e74c3c" if p > 0.5 else "#3498db" for p in probs]
    bars = ax_bar.barh(range(len(probs)), probs, color=colors, alpha=0.8)
    ax_bar.set_yticks(range(len(probs)))
    ax_bar.set_yticklabels([n.split("::")[-1] if "::" in n else n
                            for n in attr_names], fontsize=7)
    ax_bar.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)
    ax_bar.set_xlabel("Activation Probability")
    ax_bar.set_title("개념 활성화 확률 (빨강: >0.5, 파랑: ≤0.5)", fontsize=10)
    ax_bar.invert_yaxis()

    # (b) 임베딩 노름 비교
    ax_norm = fig.add_subplot(gs[1, 3:])
    z_c_norm = results["z_concept"][0].norm().item()
    z_r_norm = results["z_residual"][0].norm().item()
    z_t_norm = results["z_total"][0].norm().item()
    bar_vals = [z_c_norm, z_r_norm, z_t_norm]
    bar_labels = [f"z_concept\n‖{z_c_norm:.2f}‖",
                  f"z_residual\n‖{z_r_norm:.2f}‖",
                  f"z_total\n‖{z_t_norm:.2f}‖"]
    ax_norm.bar(bar_labels, bar_vals,
               color=["#e74c3c", "#3498db", "#2ecc71"], alpha=0.8)
    ax_norm.set_title("임베딩 L2 노름", fontsize=10)
    ax_norm.grid(True, alpha=0.3, axis="y")

    fig.suptitle(
        f"Decoding Separation 상세분석 — Sample #{idx} (Class {class_label.item()})",
        fontsize=14, fontweight="bold",
    )

    if save_dir:
        fig.savefig(str(save_dir / f"separation_detail_{idx}.png"),
                    dpi=150, bbox_inches="tight")
    plt.show()


# 3개의 대표 이미지에 대해 상세 분석
for idx in sample_indices[:3]:
    plot_single_separation(
        idx, model, test_dataset, vis_dataset, device,
        save_dir=save_dir,
    )

## 6. 정량적 분리 분석

단순 시각화 외에도, concept과 residual의 분리가 잘 되었는지 정량적으로 측정합니다:

1. **Cosine Similarity**: $z_{concept}$과 $z_{residual}$의 코사인 유사도 (낮을수록 직교)  
2. **Reconstruction Attribution**: 전체 재구성에서 각 브랜치의 기여도 비율  
3. **Concept Purity**: 잔차를 제거했을 때 개념 분류 정확도 유지 여부

In [ ]:
# === 정량적 분리 분석 ===

@torch.no_grad()
def quantitative_separation_analysis(model, dataloader, device, max_batches=20):
    """전체 데이터셋에 대해 concept-residual 분리 지표를 계산합니다."""
    model.eval()

    cos_sims = []
    concept_recon_mses = []
    residual_recon_mses = []
    full_recon_mses = []
    concept_norms = []
    residual_norms = []

    # 역정규화
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    for batch_idx, (images, concepts, _) in enumerate(tqdm(
        dataloader, desc="분리 분석", total=min(max_batches, len(dataloader))
    )):
        if batch_idx >= max_batches:
            break

        images = images.to(device)
        images_01 = (images * std + mean).clamp(0, 1)

        # 인코딩
        (z_total, z_concept, z_residual,
         concept_probs, _, _, _) = model.encode(images)

        # 디코딩 분리
        full_recon = model.decode(z_total)

        z_concept_only = model.aggregation(
            z_concept, torch.zeros_like(z_residual)
        )
        concept_recon = model.decode(z_concept_only)

        z_residual_only = model.aggregation(
            torch.zeros_like(z_concept), z_residual
        )
        residual_recon = model.decode(z_residual_only)

        # 지표 계산
        # 1. Cosine similarity between z_concept and z_residual
        min_dim = min(z_concept.shape[-1], z_residual.shape[-1])
        cos = F.cosine_similarity(
            z_concept[:, :min_dim], z_residual[:, :min_dim], dim=-1
        )
        cos_sims.append(cos.cpu())

        # 2. Reconstruction MSE
        full_mse = F.mse_loss(full_recon, images_01, reduction="none").mean(dim=[1,2,3])
        concept_mse = F.mse_loss(concept_recon, images_01, reduction="none").mean(dim=[1,2,3])
        residual_mse = F.mse_loss(residual_recon, images_01, reduction="none").mean(dim=[1,2,3])

        full_recon_mses.append(full_mse.cpu())
        concept_recon_mses.append(concept_mse.cpu())
        residual_recon_mses.append(residual_mse.cpu())

        # 3. Embedding norms
        concept_norms.append(z_concept.norm(dim=-1).cpu())
        residual_norms.append(z_residual.norm(dim=-1).cpu())

    # 결과 집계
    cos_sims = torch.cat(cos_sims)
    full_recon_mses = torch.cat(full_recon_mses)
    concept_recon_mses = torch.cat(concept_recon_mses)
    residual_recon_mses = torch.cat(residual_recon_mses)
    concept_norms = torch.cat(concept_norms)
    residual_norms = torch.cat(residual_norms)

    return {
        "cosine_similarity": cos_sims,
        "full_mse": full_recon_mses,
        "concept_mse": concept_recon_mses,
        "residual_mse": residual_recon_mses,
        "concept_norm": concept_norms,
        "residual_norm": residual_norms,
    }


metrics = quantitative_separation_analysis(model, test_loader, device)

print("=" * 60)
print("  정량적 분리 분석 결과")
print("=" * 60)
print(f"  Cosine Similarity (concept ↔ residual):")
print(f"    Mean: {metrics['cosine_similarity'].mean():.4f}")
print(f"    Std:  {metrics['cosine_similarity'].std():.4f}")
print(f"    → 0에 가까울수록 직교 (분리 성공)")
print()
print(f"  Reconstruction MSE:")
print(f"    Full:          {metrics['full_mse'].mean():.4f}")
print(f"    Concept-only:  {metrics['concept_mse'].mean():.4f}")
print(f"    Residual-only: {metrics['residual_mse'].mean():.4f}")
print()
print(f"  Embedding L2 Norms:")
print(f"    ‖z_concept‖:  {metrics['concept_norm'].mean():.4f} ± {metrics['concept_norm'].std():.4f}")
print(f"    ‖z_residual‖: {metrics['residual_norm'].mean():.4f} ± {metrics['residual_norm'].std():.4f}")
print("=" * 60)

In [ ]:
# === 정량적 분석 시각화 ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Cosine Similarity 분포
axes[0].hist(metrics["cosine_similarity"].numpy(), bins=50,
             color="#9b59b6", alpha=0.8, edgecolor="white")
axes[0].axvline(x=0, color="red", linestyle="--", linewidth=2,
                label=f"Mean={metrics['cosine_similarity'].mean():.3f}")
axes[0].axvline(x=metrics["cosine_similarity"].mean(),
                color="orange", linestyle="-", linewidth=2)
axes[0].set_title("Cosine Similarity\n(z_concept ↔ z_residual)",
                  fontweight="bold")
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# (b) Reconstruction MSE 비교
box_data = [
    metrics["full_mse"].numpy(),
    metrics["concept_mse"].numpy(),
    metrics["residual_mse"].numpy(),
]
bp = axes[1].boxplot(box_data, labels=["Full", "Concept-only", "Residual-only"],
                     patch_artist=True)
colors_box = ["#2ecc71", "#e74c3c", "#3498db"]
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title("Reconstruction MSE 비교", fontweight="bold")
axes[1].set_ylabel("MSE")
axes[1].grid(True, alpha=0.3, axis="y")

# (c) Embedding Norm 분포
axes[2].hist(metrics["concept_norm"].numpy(), bins=40, alpha=0.7,
             color="#e74c3c", label=f"Concept (μ={metrics['concept_norm'].mean():.2f})")
axes[2].hist(metrics["residual_norm"].numpy(), bins=40, alpha=0.7,
             color="#3498db", label=f"Residual (μ={metrics['residual_norm'].mean():.2f})")
axes[2].set_title("Embedding L2 Norm 분포", fontweight="bold")
axes[2].set_xlabel("L2 Norm")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("V-CoRes 정량적 분리 분석 (CUB-200)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(str(save_dir / "quantitative_separation.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 7. 개념 보간 (Concept Interpolation)

V-CoRes의 분포 기반 표현 덕분에, prior 분포 간 보간을 통해
개념이 연속적으로 변화하는 과정을 시각화할 수 있습니다.

- **잔차 고정** + 개념 보간: 배경은 유지한 채 새의 의미만 점진적으로 변화
- 이는 개념 공간이 부드럽고 해석 가능함을 증명합니다.

In [ ]:
# === 개념 보간: 두 이미지 사이의 concept을 선형 보간 ===

@torch.no_grad()
def interpolate_concepts(model, img_a, img_b, device, n_steps=8):
    """두 이미지의 concept embedding을 선형 보간하고,
    첫 번째 이미지의 residual을 고정하여 디코딩합니다.

    이를 통해 '배경은 동일, 개념만 변화'하는 과정을 관찰합니다.
    """
    model.eval()

    img_a = img_a.unsqueeze(0).to(device)
    img_b = img_b.unsqueeze(0).to(device)

    # 인코딩
    _, z_concept_a, z_residual_a, _, _, _, _ = model.encode(img_a)
    _, z_concept_b, _, _, _, _, _ = model.encode(img_b)

    # 선형 보간
    alphas = torch.linspace(0, 1, n_steps)
    interpolated_images = []

    for alpha in alphas:
        z_concept_interp = (1 - alpha) * z_concept_a + alpha * z_concept_b
        z_total = model.aggregation(z_concept_interp, z_residual_a)
        recon = model.decode(z_total)
        interpolated_images.append(recon.cpu().squeeze(0))

    return interpolated_images, alphas.numpy()


# 서로 다른 종의 새 두 장 선택
idx_a, idx_b = sample_indices[0], sample_indices[-1]
img_a_norm = test_dataset[idx_a][0]
img_b_norm = test_dataset[idx_b][0]
img_a_vis = vis_dataset[idx_a][0]
img_b_vis = vis_dataset[idx_b][0]

interp_images, alphas = interpolate_concepts(
    model, img_a_norm, img_b_norm, device, n_steps=10
)

# 시각화
fig, axes = plt.subplots(1, len(interp_images) + 2, figsize=(28, 3))

# 원본 A
axes[0].imshow(to_numpy_img(img_a_vis))
axes[0].set_title("Source A\n(원본)", fontsize=9, fontweight="bold")
axes[0].axis("off")

# 보간 결과
for i, (img, alpha) in enumerate(zip(interp_images, alphas)):
    axes[i + 1].imshow(to_numpy_img(img))
    axes[i + 1].set_title(f"α={alpha:.1f}", fontsize=8)
    axes[i + 1].axis("off")

# 원본 B
axes[-1].imshow(to_numpy_img(img_b_vis))
axes[-1].set_title("Source B\n(원본)", fontsize=9, fontweight="bold")
axes[-1].axis("off")

fig.suptitle(
    "Concept Interpolation (Residual 고정)\n"
    "A의 배경/포즈를 유지한 채, 개념을 A→B로 점진적 전환",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
fig.savefig(str(save_dir / "concept_interpolation.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 8. 잔차 교환 (Residual Swap)

두 이미지의 잔차(배경/디테일)를 교환하여,
**같은 새를 다른 배경에 놓는** 효과를 시각화합니다.

이를 통해 ResidualBranch가 배경 정보를 순수하게 흡수했음을 증명합니다.

In [ ]:
# === 잔차 교환 (Residual Swap) ===

@torch.no_grad()
def residual_swap(model, images, device):
    """여러 이미지의 잔차를 순환 교환하여 디코딩합니다.

    z_total_swap_i = AggLayer(z_concept_i, z_residual_{(i+1) mod N})
    """
    model.eval()
    images = images.to(device)
    B = images.shape[0]

    (z_total, z_concept, z_residual,
     concept_probs, _, _, _) = model.encode(images)

    # 잔차를 한 칸씩 순환 시프트
    z_residual_shifted = torch.roll(z_residual, shifts=1, dims=0)

    # 교환된 잔차로 재구성
    z_swapped = model.aggregation(z_concept, z_residual_shifted)
    swapped_recon = model.decode(z_swapped)

    # 원본 재구성
    original_recon = model.decode(z_total)

    return original_recon.cpu(), swapped_recon.cpu()


# 4장의 이미지로 잔차 교환
swap_images = torch.stack([test_dataset[i][0] for i in sample_indices[:4]])
swap_originals = torch.stack([vis_dataset[i][0] for i in sample_indices[:4]])

orig_recon, swapped_recon = residual_swap(model, swap_images, device)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
row_labels = ["원본", "원본 재구성", "잔차 교환 후\n(다음 이미지의 배경)"]

for i in range(4):
    axes[0, i].imshow(to_numpy_img(swap_originals[i]))
    axes[1, i].imshow(to_numpy_img(orig_recon[i]))
    axes[2, i].imshow(to_numpy_img(swapped_recon[i]))

    # 화살표로 잔차 출처 표시
    src = (i - 1) % 4
    axes[2, i].set_xlabel(f"← 잔차: 이미지 #{src+1}", fontsize=8, color="blue")

for row_idx, label in enumerate(row_labels):
    axes[row_idx, 0].set_ylabel(label, fontsize=10, fontweight="bold",
                                 rotation=0, labelpad=100, va="center")

for ax in axes.flat:
    ax.axis("off")

fig.suptitle(
    "Residual Swap: 개념은 유지 + 배경(잔차)만 교환\n"
    "각 열의 새는 동일하나, 배경/조명/디테일이 옆 이미지의 것으로 교체됨",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
fig.savefig(str(save_dir / "residual_swap.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 9. Memory Bank (Prior) 시각화

학습된 Memory Bank의 prior 분포 $p(z|c_k) = \mathcal{N}(\mu_k, \sigma_k^2)$를 분석합니다.

각 개념의 prior mean 간의 거리와 분산을 시각화하여,
개념 공간의 구조를 확인합니다.

In [ ]:
# === Memory Bank Prior 분석 ===

prior_mu, prior_std = model.get_concept_distributions()
prior_mu = prior_mu.cpu().numpy()
prior_std = prior_std.cpu().numpy()

attr_names = train_dataset.get_attribute_names()
short_names = [n.split("::")[-1] if "::" in n else n for n in attr_names]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# (a) Prior Mean 코사인 유사도 행렬
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_sklearn
sim_matrix = cos_sim_sklearn(prior_mu)
im = axes[0].imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_xticks(range(len(short_names)))
axes[0].set_yticks(range(len(short_names)))
axes[0].set_xticklabels(short_names, rotation=90, fontsize=7)
axes[0].set_yticklabels(short_names, fontsize=7)
axes[0].set_title("Prior Mean 코사인 유사도", fontweight="bold")
plt.colorbar(im, ax=axes[0], fraction=0.046)

# (b) Prior 평균 표준편차 (개념별 불확실성)
mean_std_per_concept = prior_std.mean(axis=1)
sorted_idx = np.argsort(mean_std_per_concept)
axes[1].barh(range(len(short_names)),
             mean_std_per_concept[sorted_idx],
             color="#e67e22", alpha=0.8)
axes[1].set_yticks(range(len(short_names)))
axes[1].set_yticklabels([short_names[i] for i in sorted_idx], fontsize=7)
axes[1].set_title("Prior σ (개념별 불확실성)", fontweight="bold")
axes[1].set_xlabel("Mean σ")
axes[1].grid(True, alpha=0.3, axis="x")

# (c) t-SNE of prior means
from sklearn.manifold import TSNE
if prior_mu.shape[0] >= 5:  # t-SNE에 최소 5개 필요
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, prior_mu.shape[0]-1))
    prior_2d = tsne.fit_transform(prior_mu)
    axes[2].scatter(prior_2d[:, 0], prior_2d[:, 1],
                    c=range(len(short_names)), cmap="tab20",
                    s=100, edgecolors="black", linewidths=0.5)
    for i, name in enumerate(short_names):
        axes[2].annotate(name, (prior_2d[i, 0], prior_2d[i, 1]),
                         fontsize=6, ha="center", va="bottom")
    axes[2].set_title("Prior Mean t-SNE", fontweight="bold")
else:
    axes[2].text(0.5, 0.5, "개념 수 부족", ha="center", va="center")

plt.suptitle("V-CoRes Memory Bank (Learnable Prior) 분석",
             fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(str(save_dir / "memory_bank_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## 10. 결론 및 해석

### 실험 요약

| 실험 | 확인할 사항 | 기대 결과 |
|------|-----------|----------|
| **Decoding Separation** | 개념/잔차 분리 품질 | Concept-only: 새 형태 유지, 배경 소실 |
| **Concept Interpolation** | 개념 공간의 연속성 | 부드러운 종(species) 전환 |
| **Residual Swap** | 잔차의 역할 | 같은 새 + 다른 배경 |
| **Quantitative Analysis** | 직교성 & 기여도 | 낮은 cosine sim, 균형 잡힌 MSE |

### 핵심 결론

1. **Information Leakage 방지**: 기존 Concept Bottleneck Model들이 개념 벡터에 배경 정보를 숨기는 문제를,
   V-CoRes는 전용 ResidualBranch가 배경을 흡수함으로써 해결합니다.

2. **개념 순수성**: Concept-only 디코딩 결과에서 배경이 사라지면,
   이는 개념 벡터가 순수하게 의미적 정보만 담고 있음을 의미합니다.

3. **분포 기반 표현**: Memory Bank의 학습된 prior 분포는 개념 간 관계와
   불확실성을 자연스럽게 포착합니다.

In [ ]:
# === 결과 JSON 저장 ===
import json

results_summary = {
    "experiment": "decoding_separation",
    "dataset": "CUB-200-2011",
    "model": "V-CoRes",
    "config": {
        "latent_dim": LATENT_DIM,
        "concept_dim": CONCEPT_DIM,
        "residual_dim": RESIDUAL_DIM,
        "num_concepts": NUM_CONCEPTS,
        "image_size": IMG_SIZE,
    },
    "metrics": {
        "cosine_similarity_mean": float(metrics["cosine_similarity"].mean()),
        "cosine_similarity_std": float(metrics["cosine_similarity"].std()),
        "full_recon_mse": float(metrics["full_mse"].mean()),
        "concept_only_mse": float(metrics["concept_mse"].mean()),
        "residual_only_mse": float(metrics["residual_mse"].mean()),
        "concept_norm_mean": float(metrics["concept_norm"].mean()),
        "residual_norm_mean": float(metrics["residual_norm"].mean()),
    },
}

results_path = save_dir.parent / "decoding_separation_results.json"
with open(str(results_path), "w") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)

print(f"결과 저장 완료: {results_path}")
print(json.dumps(results_summary, indent=2, ensure_ascii=False))